# スクレイピング準備

In [1]:
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import datetime
import os

In [61]:
Company_name = []
Industry = []
Occupation = []
Employee_size = []
position = []
date = []
total_score = []
function_score = []
use_score = []
support_score = []
cost_score = []

good_point = []
improvement_point = []
merit = []
recomend = []

# データ抽出

In [3]:
for n in range(1,15):
  time.sleep(1)
  url = f'https://it-trend.jp/expense_system/2487/review?page={n}'
  res = requests.get(url)
  soup = BeautifulSoup(res.text,"html.parser")

  profile = soup.find_all("div",attrs={"reviews mb15"})
  comment = soup.find_all("div",attrs={"reviewBox"})


  for i in range(len(profile)):
    p = profile[i].find("table")
    p = p.text.replace("\n","")
    p = p.replace(" ","")
    p = p.replace("★","")
    p = p.replace("☆","")
    Company_name.append(p[p.find("社名")+2:p.find("業種")])
    Industry.append(p[p.find("業種")+2:p.find("職種")])
    Occupation.append(p[p.find("職種")+2:p.find("従業員規模")])
    Employee_size.append(p[p.find("従業員規模")+5:p.find("立場")])
    position.append(p[p.find("立場")+2:p.find("schedule投稿日")])
    date.append(p[p.find("schedule投稿日：")+12:p.find("schedule投稿日：")+22])
    total_score.append(p[p.find("総合評価点")+5])
    function_score.append(p[p.find("機能への満足")+6])
    use_score.append(p[p.find("使いやすさ")+5])
    support_score.append(p[p.find("サポート品質")+6])
    cost_score.append(p[p.find("価格")+2])


    c = comment[i].text
    c = c.replace("\n","")
    c = c.replace(" ","")
    good_point.append(c[c.find("この製品のいい点")+8:c.find("楽楽精算の改善してほしい点")])
    improvement_point.append(c[c.find("楽楽精算の改善してほしい点")+13:c.find("楽楽精算導入で得られた効果・メリット")])
    merit.append(c[c.find("楽楽精算導入で得られた効果・メリット")+18:c.find("検討者にオススメするポイント")])
    recomend.append(c[c.find("検討者にオススメするポイント")+14:c.find("この口コミを詳しく見る")])

In [8]:
import pandas as pd
data = {}
data["投稿日"] = date
data["社名"] = Company_name
data["業種"] = Industry
data["職種"] = Occupation
data["従業員規模"] = Employee_size
data["立場"] = position
data["総合評価点"] = total_score
data["機能への満足"] = function_score
data["使いやすさ"] = use_score
data["サポート品質"] = support_score
data["価格"] = cost_score
data["この製品のいい点"] = good_point
data["改善してほしい点"] = improvement_point
data["メリット"] = merit
data["おすすめポイント"] = recomend

df = pd.DataFrame(data)

In [15]:
data2 = {}
data2["投稿日"] = date
data2["社名"] = Company_name
data2["業種"] = Industry
data2["職種"] = Occupation
data2["従業員規模"] = Employee_size
data2["立場"] = position
data2["総合評価点"] = total_score
data2["機能への満足"] = function_score
data2["使いやすさ"] = use_score
data2["サポート品質"] = support_score
data2["価格"] = cost_score

df2 = pd.DataFrame(data2)

In [32]:
data3 = {}
data3["投稿日"] = date
data3["この製品のいい点"] = good_point
data3["改善してほしい点"] = improvement_point
data3["メリット"] = merit
data3["おすすめポイント"] = recomend

df3 = pd.DataFrame(data3)

In [17]:
df2.to_csv("R精算口コミ2.csv",index="投稿日")

In [33]:
df3.to_csv("R精算口コミ3.csv")

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("ggplot")

# 形態素解析

In [51]:
# MeCabで形態素解析を行う
import MeCab
import nlplot

def mecab_text(text):

    #MeCabのインスタンスを作成（辞書はmecab-ipadic-neologdを使用）
    mecab = MeCab.Tagger('-Owakati')

    #形態素解析
    text = str(text).lower() #　追加したところ
    node = mecab.parseToNode(text)

    #形態素解析した結果を格納するリスト
    wordlist = []

    while node:
        #名詞のみリストに格納する
        if node.feature.split(',')[0] == '名詞':
            wordlist.append(node.surface)
        #形容詞を取得、elifで追加する
        #elif node.feature.split(',')[0] == '形容詞':
            wordlist.append(node.surface)
        #動詞を取得、elifで追加する
        #elif node.feature.split(',')[0] == '動詞':
            #wordlist.append(node.surface)
        node = node.next
    return wordlist

# 形態素結果をリスト化し、データフレームdf1に結果を列追加する

## この製品のいい点

In [60]:
df['words_comment'] = df['この製品のいい点'].apply(mecab_text)
npt_comment = nlplot.NLPlot(df, target_col='words_comment')

# top_nで頻出上位単語, min_freqで頻出下位単語を指定
stopwords = npt_comment.get_stopword(top_n=0, min_freq=0)

npt_comment.bar_ngram(
    title='uni-gram',
    xaxis_label='word_count',
    yaxis_label='word',
    ngram=1,
    top_n=50,
    stopwords=stopwords,
)

100%|██████████| 140/140 [00:00<00:00, 146434.55it/s]


## 改善して欲しい点

In [56]:
df['words_comment'] = df['改善してほしい点'].apply(mecab_text)
npt_comment = nlplot.NLPlot(df, target_col='words_comment')

# top_nで頻出上位単語, min_freqで頻出下位単語を指定
stopwords = npt_comment.get_stopword(top_n=0, min_freq=0)

npt_comment.bar_ngram(
    title='uni-gram',
    xaxis_label='word_count',
    yaxis_label='word',
    ngram=1,
    top_n=50,
    stopwords=stopwords,
)

100%|██████████| 140/140 [00:00<00:00, 145671.68it/s]


## メリット

In [58]:
df['words_comment'] = df['メリット'].apply(mecab_text)
npt_comment = nlplot.NLPlot(df, target_col='words_comment')

# top_nで頻出上位単語, min_freqで頻出下位単語を指定
stopwords = npt_comment.get_stopword(top_n=0, min_freq=0)

npt_comment.bar_ngram(
    title='uni-gram',
    xaxis_label='word_count',
    yaxis_label='word',
    ngram=1,
    top_n=50,
    stopwords=stopwords,
)

100%|██████████| 140/140 [00:00<00:00, 127403.46it/s]


## おすすめポイント

In [59]:
df['words_comment'] = df['おすすめポイント'].apply(mecab_text)
npt_comment = nlplot.NLPlot(df, target_col='words_comment')

# top_nで頻出上位単語, min_freqで頻出下位単語を指定
stopwords = npt_comment.get_stopword(top_n=0, min_freq=0)

npt_comment.bar_ngram(
    title='uni-gram',
    xaxis_label='word_count',
    yaxis_label='word',
    ngram=1,
    top_n=50,
    stopwords=stopwords,
)

100%|██████████| 140/140 [00:00<00:00, 75099.44it/s]
